# GEO-Bench-VLM — Download Dataset to Google Drive

Run this **once** before any eval notebook.  
The dataset is saved to `MyDrive/GEOBench/GEOBench-VLM/` and reused by all eval notebooks.  
**No GPU needed.**

In [ ]:
# ── CONFIG — only edit this cell ─────────────────────────────────────────────
DRIVE_BASE = '/content/drive/MyDrive/GEOBench'  # root folder on your Drive

In [ ]:
# ── 1. Mount Drive ────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os

DATA_PATH = f'{DRIVE_BASE}/GEOBench-VLM'
os.makedirs(DATA_PATH, exist_ok=True)

print(f'Dataset will be saved to: {DATA_PATH}')

In [ ]:
# ── 2. HuggingFace login ──────────────────────────────────────────────────────
# Add HF_TOKEN to Colab Secrets (key icon in sidebar) for unattended runs.
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get('HF_TOKEN'))
    print('Logged in via Colab secret HF_TOKEN.')
except Exception:
    from huggingface_hub import login
    login()  # fallback: interactive prompt

In [ ]:
# ── 3. Download dataset ───────────────────────────────────────────────────────
import os
from huggingface_hub import snapshot_download

single_dir = os.path.join(DATA_PATH, 'Single')
if os.path.isdir(single_dir):
    print('Dataset already extracted — skipping download.')
else:
    print('Downloading aialliance/GEOBench-VLM ...')
    snapshot_download(
        repo_id='aialliance/GEOBench-VLM',
        repo_type='dataset',
        local_dir=DATA_PATH,
    )
    print('Download complete.')

In [ ]:
# ── 4. Extract ZIP archives ───────────────────────────────────────────────────
import glob, zipfile, os

zips = glob.glob(os.path.join(DATA_PATH, '*.zip'))
if not zips:
    print('No zip files found — nothing to extract.')
else:
    for zip_path in sorted(zips):
        folder = zip_path[:-4]
        if os.path.isdir(folder):
            print(f'  Already extracted: {os.path.basename(folder)}')
        else:
            print(f'  Extracting {os.path.basename(zip_path)} ...')
            with zipfile.ZipFile(zip_path) as z:
                z.extractall(DATA_PATH)
            print(f'    Done.')
    print('All zips processed.')

In [ ]:
# ── 5. Verify structure ───────────────────────────────────────────────────────
import os

expected_splits = ['Single', 'Temporal', 'Captioning', 'Ref-Det', 'Ref-Seg']
print(f'Dataset root: {DATA_PATH}')
for split in expected_splits:
    split_path = os.path.join(DATA_PATH, split)
    qa_path    = os.path.join(split_path, 'qa.json')
    img_path   = os.path.join(split_path, 'images')
    status = '✓' if os.path.isfile(qa_path) else '✗ qa.json missing'
    imgs   = len(os.listdir(img_path)) if os.path.isdir(img_path) else 0
    print(f'  {split:<12} {status}   images: {imgs}')